# **Preparation Notebook**



---
## Setup Environment

In [34]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [35]:
# <Student to fill this section>
group_name = "Group 17"
student_name = "Chen-Tai Wang"
student_id = "25976996"

In [36]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [37]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [38]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [39]:
# <Student to fill this section and then remove this comment>

### 0.b Import Packages

In [40]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

In [41]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

---
## A. Feature Selection


## A.0 Load Data

In [42]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

### A.1 Approach 1 — Build Feature Table

In [43]:
# === Step 1: Deduplicate customer and person tables ===
customer_df_clean = customer_df.drop_duplicates(subset='customer_id', keep='first')
person_df_clean = person_df.drop_duplicates(subset='person_id', keep='first')

# === Step 2: Create target variable ===
sales_order_header_df['order_date'] = pd.to_datetime(sales_order_header_df['order_date'])

order_counts = sales_order_header_df.groupby('customer_id')['sales_order_id'].count().reset_index(name='order_count')
order_counts['is_repeat'] = (order_counts['order_count'] > 1).astype(int)

print(f"Target distribution:\n{order_counts['is_repeat'].value_counts()}")
print(f"Repeat rate: {order_counts['is_repeat'].mean():.1%}")

# === Step 3: First-order header features ===
first_order = sales_order_header_df.sort_values('order_date').groupby('customer_id').first().reset_index()
first_order['order_date'] = pd.to_datetime(first_order['order_date'])
first_order['order_month'] = first_order['order_date'].dt.month
first_order['order_dow'] = first_order['order_date'].dt.dayofweek

# === NEW: tenure_days replaces order_year ===
data_max_date = sales_order_header_df['order_date'].max()
first_order['tenure_days'] = (data_max_date - first_order['order_date']).dt.days

feat = first_order[['customer_id', 'sales_order_id', 'total_due', 'sub_total',
                     'online_order_flag', 'territory_id', 'order_month', 'order_dow', 'tenure_days']].copy()

# === Step 4: Merge target ===
feat = feat.merge(order_counts[['customer_id', 'is_repeat']], on='customer_id')

# === Step 5: Customer + Person features ===
feat = feat.merge(customer_df_clean[['customer_id', 'person_id', 'store_id']], on='customer_id', how='left')
feat = feat.merge(person_df_clean[['person_id', 'email_promotion']], on='person_id', how='left')
feat['is_store_customer'] = feat['store_id'].notna().astype(int)

# === Step 6: Territory features ===
feat = feat.merge(sales_territory_df[['territory_id', 'name']], on='territory_id', how='left')
feat.rename(columns={'name': 'territory_name'}, inplace=True)

# === Step 7: First-order detail features ===
first_detail = sales_order_detail_df.merge(
    first_order[['customer_id', 'sales_order_id']], on='sales_order_id'
)

detail_feats = first_detail.groupby('customer_id').agg(
    n_line_items=('sales_order_detail_id', 'count'),
    total_qty=('order_quantity', 'sum'),
    avg_unit_price=('unit_price', 'mean'),
    max_unit_price=('unit_price', 'max'),
    has_discount=('unit_price_discount', lambda x: int((x > 0).any()))
).reset_index()

# Product category flags
fd_prod = first_detail.merge(product_df[['product_id', 'product_subcategory_id']], on='product_id', how='left')
fd_prod = fd_prod.merge(product_sub_category_df[['product_subcategory_id', 'product_category_id']], on='product_subcategory_id', how='left')
fd_prod = fd_prod.merge(product_category_df[['product_category_id', 'name']], on='product_category_id', how='left')

cat_flags = fd_prod.groupby('customer_id')['name'].apply(lambda x: set(x.dropna())).reset_index()
cat_flags['bought_bikes'] = cat_flags['name'].apply(lambda x: int('Bikes' in x))
cat_flags['bought_accessories'] = cat_flags['name'].apply(lambda x: int('Accessories' in x))
cat_flags['bought_clothing'] = cat_flags['name'].apply(lambda x: int('Clothing' in x))
cat_flags['n_categories'] = cat_flags['name'].apply(len)

detail_feats = detail_feats.merge(
    cat_flags[['customer_id', 'bought_bikes', 'bought_accessories', 'bought_clothing', 'n_categories']],
    on='customer_id', how='left'
)

feat = feat.merge(detail_feats, on='customer_id', how='left')

# === Step 8: Missing indicators (BEFORE imputation) ===
feat['has_detail_data'] = feat['n_line_items'].notna().astype(int)
feat['has_customer_profile'] = feat['email_promotion'].notna().astype(int)

print(f"\nFeature table shape: {feat.shape}")
print(f"Columns: {feat.columns.tolist()}")
feat.head()

Target distribution:
is_repeat
0    11649
1     7470
Name: count, dtype: int64
Repeat rate: 39.1%

Feature table shape: (19119, 26)
Columns: ['customer_id', 'sales_order_id', 'total_due', 'sub_total', 'online_order_flag', 'territory_id', 'order_month', 'order_dow', 'tenure_days', 'is_repeat', 'person_id', 'store_id', 'email_promotion', 'is_store_customer', 'territory_name', 'n_line_items', 'total_qty', 'avg_unit_price', 'max_unit_price', 'has_discount', 'bought_bikes', 'bought_accessories', 'bought_clothing', 'n_categories', 'has_detail_data', 'has_customer_profile']


,customer_id,sales_order_id,total_due,sub_total,online_order_flag,territory_id,order_month,order_dow,tenure_days,is_repeat,...,total_qty,avg_unit_price,max_unit_price,has_discount,bought_bikes,bought_accessories,bought_clothing,n_categories,has_detail_data,has_customer_profile
0,00027a37-6f01-4a8f-bd81-23f2a6e7f525,ed936ddd-b6e1-48a4-b9b0-02a75206a5f0,3953.9884,3578.27,1.0,5f568b38-d738-4b7a-a27b-bc975b9084a2,9,0,1007,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,0002bd5d-7aa8-403e-aa8c-d7134d92da55,2752a541-92de-4db3-b5bd-f12afd0362f1,937.5594,848.47,1.0,c8b914d0-8c4c-498f-af9f-6954c90f45db,5,2,32,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,000421bb-5918-41c8-b95f-3bc0887876dd,3cbdcabb-db60-4ae1-b0f1-3c4d3a002864,35.6694,32.28,1.0,5f568b38-d738-4b7a-a27b-bc975b9084a2,9,4,296,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,000a3e10-bb31-43db-adf5-a4450ea0cc7e,fd8cee7f-0915-4587-9daf-2e88b25313d8,44.1779,39.98,1.0,d90fce1e-44e0-4d8f-8125-2f9ba1d18cbc,6,3,3,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,000bc388-bef0-46fd-8f64-cd76f2350ef1,e2732f2d-ef71-4f79-a481-b4e65b4319e2,8.0444,7.28,1.0,5f568b38-d738-4b7a-a27b-bc975b9084a2,1,3,163,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [44]:
feature_selection_1_insights = """
Approach 1 constructs a customer-level feature table by merging multiple data sources:
- Header features (100% coverage): total_due, sub_total, online_order_flag, territory, order_month, order_dow, order_year
- Person features (~54% coverage): email_promotion, is_store_customer
- Detail features (~48% coverage): n_line_items, total_qty, avg_unit_price, max_unit_price, has_discount, product category flags

This approach maximises information by combining all relevant datasets. Missing values in detail/person features will be handled during data cleaning.
"""

In [45]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2 — Correlation-based feature selection

In [46]:
# Check correlations with target
numeric_cols = feat.select_dtypes(include='number').columns.drop(['is_repeat'])
corrs = feat[numeric_cols].corrwith(feat['is_repeat']).sort_values(ascending=False)
print("Feature correlations with is_repeat:")
print(corrs)

chart = alt.Chart(
    corrs.reset_index().rename(columns={'index': 'Feature', 0: 'Correlation'})
).mark_bar().encode(
    x=alt.X('Correlation:Q'),
    y=alt.Y('Feature:N', sort='-x'),
    color=alt.condition(
        alt.datum.Correlation > 0,
        alt.value('steelblue'),
        alt.value('salmon')
    )
).properties(title='Feature Correlations with Target', width=500, height=400)
chart

Feature correlations with is_repeat:
tenure_days             0.652892
avg_unit_price          0.625557
max_unit_price          0.546813
bought_bikes            0.455795
sub_total               0.212386
total_due               0.210547
has_detail_data         0.198857
has_customer_profile    0.197570
is_store_customer       0.098881
has_discount            0.088975
total_qty               0.035891
order_month             0.031729
email_promotion        -0.007254
order_dow              -0.007528
n_line_items           -0.091741
online_order_flag      -0.213511
bought_clothing        -0.281450
n_categories           -0.371450
bought_accessories     -0.663682
dtype: float64


alt.Chart(...)

In [47]:
feature_selection_2_insights = """
Correlation analysis reveals strong predictive signals:
- Strongest positive: tenure_days (+0.65), avg_unit_price (+0.63), max_unit_price (+0.55), bought_bikes (+0.46)
- Strongest negative: bought_accessories (-0.66), n_categories (-0.37)
- Moderate: online_order_flag (-0.21), has_detail_data (+0.20), has_customer_profile (+0.20), is_store_customer (+0.10)

tenure_days (days between first order and dataset end) replaces order_year as a continuous measure of customer tenure.
This directly addresses the right-censoring issue — recently acquired customers have had less observation time for repeat purchases.
A continuous tenure measure provides more granular information than year buckets.

Additionally, the missing indicators (has_detail_data and has_customer_profile) show meaningful positive correlation (+0.20),
validating that structural missingness captures important business patterns rather than just random data loss.
"""

In [48]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.z Final Selection of Features

In [49]:
features_list = [
    'total_due',           # order value (header)
    'online_order_flag',   # channel (header)
    'order_month',         # seasonality (header)
    'order_dow',           # day pattern (header)
    'tenure_days',         # customer tenure in days (header)
    'is_store_customer',   # B2B flag (customer)
    'email_promotion',     # marketing preference (person)
    'territory_name',      # geography (territory)
    'n_line_items',        # order complexity (detail)
    'avg_unit_price',      # price level (detail)
    'has_discount',        # discount usage (detail)
    'bought_bikes',        # product category (detail+product)
    'bought_accessories',  # product category (detail+product)
    'bought_clothing',     # product category (detail+product)
    'n_categories',        # purchase breadth (detail+product)
    'has_detail_data',     # missing indicator for detail features
    'has_customer_profile',# missing indicator for person features
]
print(f"Selected {len(features_list)} features")

Selected 17 features


In [50]:
feature_selection_explanations = """
17 features selected spanning 5 data sources (header, customer, person, territory, detail+product).
Features cover order value, channel, temporal patterns, customer demographics, product preferences, and discount behaviour —
providing a comprehensive view of first-purchase characteristics that may predict repeat buying.

Two missing indicator features (has_detail_data, has_customer_profile) are included to distinguish between
'genuinely zero' and 'data not available' — these capture structural missingness from cross-table joins
where ~52% of customers lack matching records. This is important because tree-based models can leverage
the missingness pattern itself as a predictive signal.

tenure_days replaces order_year as a continuous measure, better capturing the right-censoring effect
where recently acquired customers have had less time to make repeat purchases.
"""

In [51]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### B.1 Fixing "Missing values in detail features"

In [52]:
# Detail features are missing for ~52% of customers (no matching sales_order_detail)
detail_cols = ['n_line_items', 'avg_unit_price', 'has_discount',
               'bought_bikes', 'bought_accessories', 'bought_clothing', 'n_categories']

print("Missing rates for detail features:")
for col in detail_cols:
    print(f"  {col}: {feat[col].isna().mean():.1%}")

print(f"\nMissing indicators already created:")
print(f"  has_detail_data: {feat['has_detail_data'].value_counts().to_dict()}")
print(f"  has_customer_profile: {feat['has_customer_profile'].value_counts().to_dict()}")

# Strategy: fill with 0 for binary flags, median for numeric
feat['n_line_items'] = feat['n_line_items'].fillna(1)  # assume 1 item if no detail
feat['avg_unit_price'] = feat['avg_unit_price'].fillna(feat['avg_unit_price'].median())
feat['has_discount'] = feat['has_discount'].fillna(0)
feat['bought_bikes'] = feat['bought_bikes'].fillna(0)
feat['bought_accessories'] = feat['bought_accessories'].fillna(0)
feat['bought_clothing'] = feat['bought_clothing'].fillna(0)
feat['n_categories'] = feat['n_categories'].fillna(0)

print("\nAfter imputation — remaining nulls:")
print(feat[detail_cols].isnull().sum())

Missing rates for detail features:
  n_line_items: 51.7%
  avg_unit_price: 51.7%
  has_discount: 51.7%
  bought_bikes: 51.7%
  bought_accessories: 51.7%
  bought_clothing: 51.7%
  n_categories: 51.7%

Missing indicators already created:
  has_detail_data: {0: 9887, 1: 9232}
  has_customer_profile: {0: 9958, 1: 9161}

After imputation — remaining nulls:
n_line_items          0
avg_unit_price        0
has_discount          0
bought_bikes          0
bought_accessories    0
bought_clothing       0
n_categories          0
dtype: int64


In [53]:
data_cleaning_1_explanations = """
Approximately 52% of customers lack matching records in sales_order_detail, resulting in missing values for detail-derived features.
This is structural missingness (not random), as certain orders simply have no detail records in the dataset.

Two missing indicator features (has_detail_data, has_customer_profile) were created BEFORE imputation to preserve the missingness signal.
This allows the model to distinguish between 'customer had no discount' vs 'we have no data for this customer' —
a distinction with different business implications.

Imputation strategy (applied after indicators):
- Binary flags (has_discount, bought_bikes, bought_accessories, bought_clothing): filled with 0 (conservative assumption)
- n_line_items: filled with 1 (minimum possible for any order)
- avg_unit_price: filled with median to avoid extreme value bias
- n_categories: filled with 0
"""

In [54]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "Missing values in email_promotion"

In [55]:
print(f"email_promotion missing: {feat['email_promotion'].isna().sum()} ({feat['email_promotion'].isna().mean():.1%})")
print(f"Distribution (non-null):\n{feat['email_promotion'].value_counts()}")

# Fill with mode (0 = no promotion) since it's the most common value
feat['email_promotion'] = feat['email_promotion'].fillna(0)

print(f"\nAfter imputation:\n{feat['email_promotion'].value_counts()}")

email_promotion missing: 9958 (52.1%)
Distribution (non-null):
email_promotion
0.0    5160
1.0    2298
2.0    1703
Name: count, dtype: int64

After imputation:
email_promotion
0.0    15118
1.0     2298
2.0     1703
Name: count, dtype: int64


In [56]:
data_cleaning_2_explanations = """
Created is_high_value as a binary indicator for orders above the training set median ($596.69). This captures the bimodal distribution observed in EDA —
high-value customers (primarily bike buyers) likely exhibit different repeat purchase behaviour than low-value customers (accessories/clothing).
The median threshold is computed from training data only to prevent data leakage.
"""

In [57]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "Missing territory_name"

In [58]:
print(f"territory_name missing: {feat['territory_name'].isna().sum()}")

# Fill remaining territory_name NaN with 'Unknown'
feat['territory_name'] = feat['territory_name'].fillna('Unknown')
print(f"Territory distribution:\n{feat['territory_name'].value_counts()}")

territory_name missing: 9887
Territory distribution:
territory_name
Unknown           9887
Australia         3625
United Kingdom    1951
France            1844
Germany           1812
Name: count, dtype: int64


In [59]:
data_cleaning_3_explanations = """
A small number of customers have missing territory_name after the join. These are filled with 'Unknown' as a separate category,
since dropping them would lose data and imputing a specific territory would be misleading.
"""

In [60]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

In [61]:
# Prepare the dataset for splitting
# Keep only selected features + target
model_df = feat[features_list + ['is_repeat']].copy()

# Split: 70% train, 15% validation, 15% test (stratified)
from sklearn.model_selection import train_test_split

train_val, testing_df_clean = train_test_split(model_df, test_size=0.15, random_state=42, stratify=model_df['is_repeat'])
training_df_clean, validation_df_clean = train_test_split(train_val, test_size=0.176, random_state=42, stratify=train_val['is_repeat'])
# 0.176 of 0.85 ≈ 0.15 of total

print(f"Train: {len(training_df_clean)} ({len(training_df_clean)/len(model_df):.1%})")
print(f"Val:   {len(validation_df_clean)} ({len(validation_df_clean)/len(model_df):.1%})")
print(f"Test:  {len(testing_df_clean)} ({len(testing_df_clean)/len(model_df):.1%})")

for name, df_split in [('Train', training_df_clean), ('Val', validation_df_clean), ('Test', testing_df_clean)]:
    print(f"{name} target: {df_split['is_repeat'].value_counts().to_dict()}")

Train: 13390 (70.0%)
Val:   2861 (15.0%)
Test:  2868 (15.0%)
Train target: {0: 8159, 1: 5231}
Val target: {0: 1743, 1: 1118}
Test target: {0: 1747, 1: 1121}


---
## C. Feature Engineering

In [62]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df_clean.copy()
  validation_df_eng = validation_df_clean.copy()
  testing_df_eng = testing_df_clean.copy()
except Exception as e:
  print(e)

### C.1 New Feature "log_total_due"



In [63]:
for df_split in [training_df_eng, validation_df_eng, testing_df_eng]:
    df_split['log_total_due'] = np.log1p(df_split['total_due'])

print(f"log_total_due stats (train):")
print(training_df_eng['log_total_due'].describe())

log_total_due stats (train):
count    13390.000000
mean         5.694223
std          2.112588
min          1.261440
25%          3.810608
50%          6.393071
75%          7.736272
max         12.141475
Name: log_total_due, dtype: float64


In [64]:
feature_engineering_1_explanations = """
Created log_total_due = log(1 + total_due) to handle the heavy right-skew in order values. This transformation compresses the long tail and creates a more normally distributed feature,
which improves performance for distance-based algorithms and regularised models. The log-transformed feature better separates the two customer segments (small accessory orders vs. large bike orders).
"""

In [65]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### C.3 New Feature "is_high_value" (based on training median)



In [66]:
median_total = training_df_eng['total_due'].median()
print(f"Training median total_due: {median_total:.2f}")

for df_split in [training_df_eng, validation_df_eng, testing_df_eng]:
    df_split['is_high_value'] = (df_split['total_due'] > median_total).astype(int)

print(f"is_high_value distribution (train):\n{training_df_eng['is_high_value'].value_counts()}")

Training median total_due: 596.69
is_high_value distribution (train):
is_high_value
0    6749
1    6641
Name: count, dtype: int64


In [67]:
feature_engineering_2_explanations = """
Created is_high_value as a binary indicator for orders above the training set median ($X). This captures the bimodal distribution observed in EDA —
high-value customers (primarily bike buyers) likely exhibit different repeat purchase behaviour than low-value customers (accessories/clothing).
The median threshold is computed from training data only to prevent data leakage.
"""

In [68]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### C.4 New Feature "is_weekend_order"



In [69]:
for df_split in [training_df_eng, validation_df_eng, testing_df_eng]:
    df_split['is_weekend_order'] = (df_split['order_dow'] >= 5).astype(int)

print(f"is_weekend_order distribution (train):\n{training_df_eng['is_weekend_order'].value_counts()}")

is_weekend_order distribution (train):
is_weekend_order
0    9530
1    3860
Name: count, dtype: int64


In [70]:
feature_engineering_3_explanations = """
Created is_weekend_order as a binary indicator for orders placed on Saturday or Sunday (order_dow >= 5). Weekend vs weekday purchasing may reflect different customer motivations —
weekend shoppers might be more casual/impulsive, while weekday shoppers could be more deliberate, potentially affecting repeat rates.
"""

In [71]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

---
## D. Data Preparation for Modeling

### D.1 Split Datasets


In [72]:
# Define target
target_col = 'is_repeat'

# Identify feature columns (exclude IDs and original total_due since we have log version)
drop_cols = ['is_repeat', 'total_due']
feature_cols = [c for c in training_df_eng.columns if c not in drop_cols]

X_train = training_df_eng[feature_cols].copy()
y_train = training_df_eng[[target_col]].copy()

X_val = validation_df_eng[feature_cols].copy()
y_val = validation_df_eng[[target_col]].copy()

X_test = testing_df_eng[feature_cols].copy()
y_test = testing_df_eng[[target_col]].copy()

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (13390, 19), y_train: (13390, 1)
X_val: (2861, 19), y_val: (2861, 1)
X_test: (2868, 19), y_test: (2868, 1)


In [73]:
data_splitting_explanations = """
Data is split 70/15/15 into train/validation/test sets using stratified sampling to preserve the target class ratio (~61/39).
The validation set is used for hyperparameter tuning and model selection, while the test set is held out for final unbiased evaluation.
Stratification ensures each split reflects the original class distribution.
"""

In [74]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### D.2 Data Transformation — Encoding Categorical Features


In [75]:
# One-hot encode territory_name
X_train = pd.get_dummies(X_train, columns=['territory_name'], drop_first=True, dtype=int)
X_val = pd.get_dummies(X_val, columns=['territory_name'], drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=['territory_name'], drop_first=True, dtype=int)

# Align columns (val/test may have missing categories)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"After encoding — X_train columns: {X_train.columns.tolist()}")
print(f"Shape: {X_train.shape}")

After encoding — X_train columns: ['online_order_flag', 'order_month', 'order_dow', 'tenure_days', 'is_store_customer', 'email_promotion', 'n_line_items', 'avg_unit_price', 'has_discount', 'bought_bikes', 'bought_accessories', 'bought_clothing', 'n_categories', 'has_detail_data', 'has_customer_profile', 'log_total_due', 'is_high_value', 'is_weekend_order', 'territory_name_France', 'territory_name_Germany', 'territory_name_United Kingdom', 'territory_name_Unknown']
Shape: (13390, 22)


In [76]:
data_transformation_1_explanations = """
territory_name is one-hot encoded with drop_first=True to avoid multicollinearity (4 territories → 3 binary columns).
Validation and test sets are aligned to training columns to ensure consistent feature dimensions. Any missing categories in val/test are filled with 0.
"""

In [77]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### D.3 Data Transformation — Scaling Numeric Features

In [78]:
numeric_features = ['log_total_due', 'avg_unit_price', 'n_line_items', 'n_categories', 'tenure_days']
# Only scale continuous features; leave binary flags as-is

scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

print(f"Scaled feature means (train):\n{X_train[numeric_features].mean()}")
print(f"\nScaled feature stds (train):\n{X_train[numeric_features].std()}")

Scaled feature means (train):
log_total_due    -4.245214e-18
avg_unit_price    1.273564e-16
n_line_items      3.290041e-17
n_categories      5.306518e-19
tenure_days      -1.082530e-16
dtype: float64

Scaled feature stds (train):
log_total_due     1.000037
avg_unit_price    1.000037
n_line_items      1.000037
n_categories      1.000037
tenure_days       1.000037
dtype: float64


In [79]:
data_transformation_2_explanations = """
StandardScaler is applied to continuous numeric features (log_total_due, avg_unit_price, n_line_items, n_categories) to standardise them to zero mean and unit variance.
This is important for algorithms sensitive to feature scale (e.g., logistic regression, SVM, KNN).

Binary features (online_order_flag, is_store_customer, has_discount, bought_bikes, etc.) are left unscaled as they are already 0/1.

The scaler is fit ONLY on training data and applied to validation/test to prevent data leakage.
"""

In [80]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

---
## E. Save Datasets

> Do not change this code

In [81]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)